<a href="https://colab.research.google.com/github/craljimenez/An-lisis_y_Visualizacion_de_Datos_para_la_Toma_de_Decisiones/blob/main/clases/semana04-clase07-pandas-dataframes-diagnostico-calidad/notebooks/lab-pandas-series-dataframe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio: Pandas, Series, DataFrames y diagnóstico de calidad
### Semana 4 · Clase 7
**Curso:** Análisis y Visualización de Datos para la Toma de Decisiones (Esumer)

Desde la Clase 4 venimos usando `DataFrame`s de pandas para responder preguntas de negocio (filtrar, resumir, calcular). Lo que no hemos hecho todavía es **formalizar qué es exactamente** un `DataFrame`, de qué está hecho por dentro, y qué herramientas tiene pandas para revisar si un dataset está "sano" antes de confiar en él.

Hoy hacemos dos cosas:
1. Formalizamos **`Series`** y **`DataFrame`**: qué son, en qué se diferencian, cómo reconocer cuál tienes en las manos, y sus métodos más importantes.
2. Aprendemos a hacer un **diagnóstico de calidad**: nulos, duplicados, tipos incorrectos e inconsistencias, los mismos problemas que en la Clase 4 vimos aparecer "por accidente" al leer un archivo, pero ahora buscados **a propósito y de forma sistemática**.

## 1. La `Series`: el bloque de construcción de pandas

Una **`Series`** es una estructura de **una sola dimensión**: una columna de datos, con un **índice** (una etiqueta para cada valor) al lado. Es el ladrillo con el que está construido todo `DataFrame`.

Se parece a la lista de Python que usaste en la Clase 3, pero con una diferencia clave: cada valor tiene una **etiqueta** (el índice), no solo una posición.

In [ ]:
import pandas as pd

edades = pd.Series([28, 34, 19, 45])
edades

A la izquierda ves el **índice** (por defecto, `0, 1, 2, 3`, igual que las posiciones de una lista); a la derecha, los **valores**. Abajo, pandas te dice el `dtype` (tipo de dato) de toda la Serie.

También puedes construir una `Series` con un índice propio, por ejemplo usando nombres en vez de posiciones. Esto se parece más a un diccionario de Python (Clase 3) que a una lista:

In [ ]:
edades_por_socio = pd.Series(
    [28, 34, 19, 45],
    index=["Camila", "Andrés", "Valentina", "Santiago"]
)
edades_por_socio

### Atributos y métodos básicos de una `Series`

Una `Series` guarda tres cosas: sus **valores**, su **índice** y su **tipo de dato**, y trae métodos para resumirla de un vistazo.

In [ ]:
print("Valores:", edades_por_socio.values)
print("Índice:", edades_por_socio.index)
print("Tipo de dato:", edades_por_socio.dtype)

In [ ]:
edades_por_socio.describe()

`describe()` en una `Series` numérica resume conteo, promedio, desviación estándar, mínimo, percentiles y máximo, todo lo que viste en la Clase 2 de estadística descriptiva, ahora calculado con un solo método.

Otros métodos muy usados sobre una `Series`:

In [ ]:
ciudades = pd.Series(["Medellín", "Bogotá", "Medellín", "Cali", "Medellín", "Bogotá"])

print(ciudades.unique())        # valores distintos, sin repetir
print(ciudades.value_counts())  # cuántas veces aparece cada valor
print(ciudades.nunique())       # cuántos valores distintos hay

Actividad: crea una `Series` con los precios de 5 productos que tú elijas, y usa `.mean()`, `.max()` y `.min()` para resumirla.

In [ ]:
# TODO: crea tu Series de precios y calcula mean(), max() y min()


## 2. El `DataFrame`: una tabla hecha de `Series`

Un **`DataFrame`** es una tabla de dos dimensiones (filas y columnas). La idea clave que casi nunca se dice explícitamente: **cada columna de un `DataFrame` es, por dentro, una `Series`**, y todas comparten el mismo índice (las filas).

In [ ]:
socios_chico = pd.DataFrame({
    "nombre": ["Camila", "Andrés", "Valentina"],
    "ciudad": ["Medellín", "Bogotá", "Medellín"],
    "edad": [28, 34, 19],
})
socios_chico

### El detalle que confunde a todo el mundo al principio: `df["col"]` vs. `df[["col"]]`

Con **un corchete**, sacas la columna como `Series`. Con **dos corchetes** (una lista con un solo nombre adentro), sacas esa misma columna pero como `DataFrame` de una sola columna. Compruébalo con `type()`:

In [ ]:
una_columna = socios_chico["edad"]
print(type(una_columna))
una_columna

In [ ]:
misma_columna_como_tabla = socios_chico[["edad"]]
print(type(misma_columna_como_tabla))
misma_columna_como_tabla

| Sintaxis | Qué devuelve | Dimensión |
|---|---|---|
| `df["col"]` | `Series` | 1D (una columna suelta) |
| `df[["col"]]` | `DataFrame` | 2D (una tabla de una sola columna) |
| `df[["col1", "col2"]]` | `DataFrame` | 2D (una tabla con esas columnas) |

¿Cuándo importa la diferencia? Cuando necesitas seguir usando métodos propios de `DataFrame` (como `.describe()` con varias columnas, o unir con otra tabla más adelante) en vez de los de `Series`. Si alguna vez un método te da un error raro tipo *"Series object has no attribute..."*, revisa primero si sacaste una `Series` cuando en realidad necesitabas un `DataFrame` de una columna.

Actividad: de `socios_chico`, saca la columna `ciudad` dos veces: una como `Series` y otra como `DataFrame`. Confirma el tipo de cada una con `type()`.

In [ ]:
# TODO: extrae "ciudad" como Series y como DataFrame; confirma con type()


## 3. Explorar un `DataFrame` real: `membresias_gimnasio.csv`

De aquí en adelante trabajamos con datos reales (bueno, realistas): las inscripciones del **Gimnasio Vitalis**, un dataset con los mismos problemas que te vas a encontrar en datasets de verdad (nulos, duplicados, tipos raros e inconsistencias) sembrados a propósito para que aprendas a detectarlos.

Descarga `membresias_gimnasio.csv` desde la carpeta `datos/` de esta clase, en el repositorio del curso, y súbelo a Colab.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
socios = pd.read_csv("membresias_gimnasio.csv")
socios.head()

### El "checklist" de primera vista

Antes de responder cualquier pregunta de negocio sobre un dataset nuevo, conviene mirarlo con este checklist rápido; lo usarás siempre, con cualquier dataset que te encuentres de aquí en adelante.

In [ ]:
print("Filas y columnas:", socios.shape)
print("\nColumnas:", list(socios.columns))

In [ ]:
socios.dtypes

In [ ]:
socios.info()

`.info()` junta varias cosas: el tipo de cada columna, cuántos valores **no nulos** tiene cada una (compáralo con el total de filas del `.shape` de arriba) y cuánta memoria ocupa la tabla. Es normalmente la primera orden que deberías ejecutar sobre cualquier dataset nuevo.

In [ ]:
socios.sample(5)  # 5 filas al azar, distintas cada vez que la ejecutas

`.sample(n)` es un complemento útil de `.head()`: `.head()` siempre te muestra el principio (que a veces está "arreglado" o no representa bien al resto), mientras que `.sample()` te da una vista más honesta de toda la tabla.

In [ ]:
socios.describe()

Por defecto, `.describe()` solo resume las columnas **numéricas** (aquí, `id_socio` y quizás alguna más, según cómo haya quedado leída). Para ver también las columnas de texto, se usa `include="all"`:

In [ ]:
socios.describe(include="all")

Para las columnas de texto, `describe(include="all")` muestra otra cosa: `unique` (cuántos valores distintos hay), `top` (el valor más frecuente) y `freq` (cuántas veces aparece ese valor más frecuente). Fíjate en la columna `ciudad`: ¿cuántos valores únicos dice que tiene? Guarda ese número en la cabeza: vamos a volver a él en la Sección 5, y te vas a sorprender.

Actividad: mirando el resultado de `.describe(include="all")`, ¿qué columna(s) tienen menos de 54 en `count`? Eso ya es una primera pista de dónde hay datos faltantes, antes incluso de contarlos formalmente.

## 4. Selección por columnas y filas: `.loc` y `.iloc`

Desde la Clase 4 ya filtras filas con `df[condición]`. Pandas tiene además dos formas más explícitas de seleccionar, que te dan control sobre **filas y columnas al mismo tiempo**:

- **`.loc[filas, columnas]`**: selección **por etiqueta** (nombre de columna, valor de índice, o una condición).
- **`.iloc[filas, columnas]`**: selección **por posición** (como en una lista: 0, 1, 2...).

In [ ]:
socios.loc[0]  # la fila con índice 0 (todas sus columnas)

In [ ]:
socios.loc[0:2, ["nombre", "ciudad"]]  # filas 0 a 2, solo dos columnas

In [ ]:
socios.iloc[0:3, 0:2]  # las primeras 3 filas, las primeras 2 columnas, por posición

Donde `.loc` se vuelve realmente útil es combinado con una condición, exactamente como el `WHERE` de SQL en la Clase 5, pero ahora eligiendo también qué columnas quieres ver, en la misma instrucción:

In [ ]:
socios.loc[socios["ciudad"] == "Medellín", ["nombre", "ciudad", "plan"]]

Compáralo con lo que ya sabías: `socios[socios["ciudad"] == "Medellín"]` te da **todas** las columnas; `.loc[condición, columnas]` te deja elegir cuáles, en un solo paso.

Actividad: usa `.loc` para obtener `nombre` y `telefono` de los socios cuyo `plan` sea `"Anual"`.

In [ ]:
# TODO: usa .loc para nombre y telefono donde plan == "Anual"


## 5. El índice: fijar una columna como identificador

Fíjate en el número que aparece a la izquierda de cada fila cada vez que muestras `socios`: eso es el **índice**. Hasta ahora ha sido el que pandas pone por defecto (`0, 1, 2, 3...`, un `RangeIndex`), el mismo que ya usaste en la Sección 1 con las `Series` y en la Sección 4 con `.loc[0]`.

Pero el índice **no tiene que ser ese contador automático**: cuando una columna ya funciona como identificador natural de cada fila (como `id_socio` aquí), tiene sentido convertirla en el índice de verdad.

In [ ]:
socios.index

### Fijar una columna como índice: `set_index()`

`set_index("columna")` convierte esa columna en el índice del `DataFrame`. Ojo: esto **no modifica** `socios`, sino que devuelve una copia nueva (a menos que uses `inplace=True`, que en general es mejor evitar, para no perder de vista qué versión de la tabla estás usando).

In [ ]:
socios_por_id = socios.set_index("id_socio")
socios_por_id.head()

Fíjate en dos cosas: la columna `id_socio` **ya no aparece como columna normal** (ahora es el índice, se muestra aparte, a la izquierda) y la tabla ya no tiene el contador `0, 1, 2...`.

In [ ]:
socios_por_id.columns

### Lo que esto implica: `.loc` ahora busca por ese identificador

Con `id_socio` como índice, `.loc[]` ya no busca por posición: busca por el **valor de `id_socio`**, sin importar en qué fila física quedó guardado. Eso es justo lo que lo hace útil: puedes pedir el socio número 3 sin saber (ni que te importe) en qué posición quedó dentro de la tabla.

In [ ]:
socios_por_id.loc[3]

Compáralo ahora con `.iloc[3]`, que ignora por completo el índice y busca **por posición**, igual que antes:

In [ ]:
socios_por_id.iloc[3]

Son personas distintas. `.loc[3]` te dio el socio **con identificador 3**; `.iloc[3]` te dio lo que sea que haya quedado en la **posición 3** de la tabla (que, como los datos vienen desordenados, no tiene por qué coincidir). Quédate con esta idea: **`.loc` es por etiqueta, `.iloc` es por posición**: nunca asumas que dan lo mismo.

### El riesgo de un índice con valores repetidos

¿Qué pasa si la columna que conviertes en índice **tiene valores repetidos**? Pruébalo con un `id_socio` que se repita:

In [ ]:
socios_por_id.loc[4]

En vez de una fila (una `Series`), `.loc[4]` te devolvió **dos filas** (un `DataFrame`), todas las que tienen `id_socio = 4`. Si en tu cabeza `id_socio` era un identificador único, esto es una señal de alarma: significa que hay **filas duplicadas** en `socios`, algo que vamos a confirmar formalmente en un momento, en la Sección 6.2.

Esta es la razón práctica para fijar un índice con cuidado: un índice pensado para identificar filas de forma única deja de servir para eso si el dato de origen tiene duplicados o errores.

### Deshacer el cambio: `reset_index()`

`reset_index()` hace lo contrario: saca el índice actual y lo vuelve a poner como columna normal, reemplazándolo por el `RangeIndex` de siempre.

In [ ]:
socios_por_id.reset_index().head()

¿Cuándo vale la pena fijar un índice, entonces? Cuando la columna es un identificador **único y confiable** y vas a buscar filas por él muy seguido. Cuando no estás seguro de que sea único (como aquí, antes de auditar duplicados), es más seguro dejar el índice por defecto y seguir filtrando con `.loc[condición]` o `df[condición]`, como ya sabías.

Actividad: fija `nombre` como índice en una copia de `socios` (sin sobrescribir `socios`), y usa `.loc[]` para buscarte a ti mismo un nombre cualquiera de la tabla. Antes de hacerlo, ¿crees que `nombre` es un identificador seguro para usar como índice en este dataset? ¿Por qué sí o por qué no?

In [ ]:
# TODO: crea socios_por_nombre = socios.set_index(...) y prueba .loc[] con un nombre


## 6. Diagnóstico de calidad: nulos, duplicados, tipos e inconsistencias

Ahora sí, el corazón de la clase de hoy: revisar sistemáticamente qué tan "sano" está `socios` antes de confiar en él para cualquier análisis.

### 6.1 Nulos

In [ ]:
socios.isna().sum()

`.isna()` marca `True`/`False` en cada celda (¿está vacía?); `.sum()` sobre eso cuenta cuántos `True` hay por columna, porque pandas trata `True` como `1` y `False` como `0`. Es más útil ver el **porcentaje**, sobre todo si comparas datasets de distinto tamaño:

In [ ]:
(socios.isna().mean() * 100).round(1)

### 6.2 Duplicados

In [ ]:
print("Filas duplicadas:", socios.duplicated().sum())
socios[socios.duplicated(keep=False)].sort_values("id_socio")

`.duplicated()` marca `True` en las filas que son una copia **exacta** de una fila anterior (`keep=False` en el ejemplo de arriba, en cambio, marca **todas** las copias, la original y las repetidas, para que puedas verlas juntas). Para quedarte solo con una copia de cada una, se usa `.drop_duplicates()`.

### 6.3 Tipos incorrectos

Vuelve a mirar `socios.dtypes` de la Sección 3. Dos columnas deberían llamarte la atención:

- **`telefono`** quedó como `float64` (número decimal). Pero un teléfono **no es una cantidad con la que sumas o promedias**: es texto que por casualidad solo tiene dígitos. Al leerlo como número, corres el riesgo de perder ceros a la izquierda o verlo en notación científica.

In [ ]:
socios[["nombre", "telefono"]].head(3)

Justo ese problema: `3031045678` se muestra como `3.031046e+09` (notación científica): el teléfono real quedó irreconocible. Esa es una razón muy concreta para **convertir `telefono` a texto** en la limpieza de la Clase 8, no dejarlo como número.

- **`edad`** quedó como texto (`str`), cuando debería ser un número. ¿Por qué, si la mayoría de sus valores se ven como números? Porque unos pocos valores **no lo son**, y eso le alcanza a pandas para no tratar toda la columna como numérica.

In [ ]:
edad_numerica = pd.to_numeric(socios["edad"], errors="coerce")
problemas_edad = socios.loc[edad_numerica.isna() & socios["edad"].notna(), ["nombre", "edad"]]
problemas_edad

Usamos el mismo patrón de la Clase 4: `pd.to_numeric(..., errors="coerce")` convierte lo que sí es número y vuelve `NaN` lo que no. Filtrando por "se volvió `NaN`, pero el dato original **no** estaba vacío" (`edad_numerica.isna() & socios["edad"].notna()`), aislamos justo los valores raros: texto disfrazado de número, que `.isna()` por sí solo **nunca te habría mostrado**, porque técnicamente no son nulos.

### 6.4 Inconsistencias de texto

Recuerda la pregunta que dejamos pendiente en la Sección 3: ¿cuántos valores únicos tiene `ciudad`? Míralo con `.value_counts()`:

In [ ]:
socios["ciudad"].value_counts()

Esa lista tiene muchas más "ciudades" de las que realmente existen: `Medellín`, `medellín`, `MEDELLIN`, `Medellin` y `" Medellín"` (con espacio al inicio) son, para una persona, **la misma ciudad**, pero para pandas son 5 valores distintos, porque compara el texto letra por letra, tildes, mayúsculas y espacios incluidos.

El mismo problema aparece en `activo`:

In [ ]:
socios["activo"].value_counts()

`Sí`, `SI`, `si`, `"Si "` (con espacio) y `1` deberían ser un solo valor ("verdadero"); `No`, `NO`, `no` y `0`, el otro. Ahora mismo son 9 valores distintos.

Este tipo de inconsistencia es traicionero porque **no genera ningún error**: el código corre, `.value_counts()` no se queja, y solo lo notas si te detienes a mirar los valores únicos de cada columna de texto. Por eso vale la pena revisar `.value_counts()` de toda columna categórica antes de usarla para agrupar o filtrar.

*(Cómo corregirlo de verdad, con `.str.strip()`, `.str.lower()`, mapear valores a una categoría estándar, es exactamente el tema de la Clase 8. Hoy nos quedamos en identificarlo y dejarlo documentado.)*

Actividad: `plan` es otra columna de texto. Revisa su `.value_counts()`: ¿tiene el mismo problema de inconsistencia que `ciudad` y `activo`, o está limpia?

In [ ]:
# TODO: revisa socios["plan"].value_counts()


## 7. Reto: auditoría de calidad sobre un dataset de tu interés

Ahora te toca a ti, con datos que **tú elijas**: puede ser un Excel/CSV que ya tengas (notas, gastos, un dataset de tu trabajo, algo de [Kaggle](https://www.kaggle.com/datasets) o [datos.gov.co](https://www.datos.gov.co/)), lo importante es que te interese de verdad.

Sube tu archivo con `files.upload()` (o `pd.read_excel()` si es un `.xlsx`) y completa el checklist de auditoría, con lo que aprendiste hoy.

In [ ]:
# Sube tu archivo
from google.colab import files

uploaded = files.upload()

In [ ]:
# TODO: lee tu archivo en un DataFrame (usa el nombre que tenga sentido para tus datos)
mis_datos = None

**Paso 1 (Primera vista):** `shape`, `columns`, `dtypes`, `head()`, `sample()`.

In [ ]:
# TODO: primera vista de mis_datos


**Paso 2 (Resumen general):** `info()` y `describe(include="all")`.

In [ ]:
# TODO: info() y describe(include="all")


**Paso 3 (Nulos):** ¿cuántos hay por columna, y qué porcentaje representan?

In [ ]:
# TODO: cuenta nulos por columna (cantidad y %)


**Paso 4 (Duplicados):** ¿hay filas exactamente repetidas? ¿Cuántas?

In [ ]:
# TODO: cuenta filas duplicadas


**Paso 5 (Tipos incorrectos):** revisa `dtypes`. ¿Alguna columna que debería ser número quedó como texto (o viceversa)? Si es así, usa `pd.to_numeric(..., errors="coerce")` como en la Sección 6.3 para encontrar los valores problemáticos.

In [ ]:
# TODO: revisa tipos y encuentra valores problemáticos si aplica


**Paso 6 (Inconsistencias):** elige al menos una columna de texto/categórica y revisa su `.value_counts()`. ¿Hay variantes de un mismo valor (mayúsculas, tildes, espacios)?

In [ ]:
# TODO: value_counts() de una columna categórica de tu dataset


### Paso 7: Ficha de auditoría de calidad (entregable)

Completa esta ficha en la celda de markdown de abajo (doble clic para editar), con lo que encontraste. Esta ficha es tu insumo directo para la Clase 8, donde vamos a **corregir** varios de estos problemas.

> **Dataset:** *(nombre del archivo y de dónde lo sacaste)*
>
> **Tamaño:** *(filas × columnas)*
>
> **Principales problemas encontrados** *(marca los que apliquen y agrega el detalle)*:
> - [ ] Nulos, columna(s): ______, % aproximado: ______
> - [ ] Duplicados, cantidad: ______
> - [ ] Tipos incorrectos, columna(s): ______, tipo actual vs. esperado: ______
> - [ ] Inconsistencias de texto, columna(s): ______, variantes encontradas: ______
>
> **Reglas de limpieza propuestas** *(en palabras, 3 a 5 reglas; no hace falta el código todavía)*:
> 1. ...
> 2. ...
> 3. ...

## Cierre

Hoy formalizaste `Series` y `DataFrame`, aprendiste a explorar un dataset nuevo de forma sistemática (`shape`, `info()`, `describe()`), a seleccionar con `.loc`/`.iloc`, y a **diagnosticar** cuatro tipos de problemas de calidad: nulos, duplicados, tipos incorrectos e inconsistencias. Guarda una copia de este notebook (y tu ficha de auditoría) en tu Google Drive antes de cerrar Colab.

En la **Clase 8: Transformación y enriquecimiento de datos** volvemos exactamente sobre los problemas que hoy solo diagnosticaste: ahí sí vamos a **corregirlos** (columnas calculadas, renombrar variables, categorización, `.str.strip()`/`.str.lower()` para las inconsistencias de texto, y convertir `telefono` y `edad` a su tipo correcto) y a construir un pipeline de limpieza reproducible.